# AMADS Coding Notebooks

## Upper and Lower Envelope

Here we extract 'envelope' filtering related to skyline and valleyline reductions of musical data.

Context:
- Strict skyline/valleyline functions filter for the highest/lowest note at any point.
    - Here, the envelope functions reduce that data slightly further.
- Regression fits through the *middle* of the data, these envelopes:
    - pass through **actual data points**
    - support an **arbitrary number of direction changes**
    - exclude only points that genuinely dip inside the boundary

---

**By (author/s):** Mark Gotham

**For:** Attached to the
[AMADS code library](https://github.com/music-computing/amads/) and
["Keeping Score" book](https://doi.org/10.5281/zenodo.14938027),
but open to all.

**Licence:** MIT.

**Colour key:**
- <font color='green'> Green is for a block of information.
- <font color='purple'> Purple is for an exercise.
- <font color='crimson'> Crimson is for the solution to the exercise above it.m

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from amads.polyphony import envelope

## <font color='green'> Plotting helper and toy examples

In [ ]:
def plot_envelopes(source, title='', show_valley=True, tolerance=0.0):
    """
    Plot raw data, skyline, and (optionally) valleyline for any supported input.
    """
    # Coerce input to points so we can plot the raw data
    raw   = envelope._to_points(source)
    xs    = [p[0] for p in raw]
    ys    = [p[1] for p in raw]

    sky  = envelope.skyline_envelope(source, tolerance=tolerance)
    sky_x, sky_y = zip(*sky) if sky else ([], [])

    fig, ax = plt.subplots(figsize=(12, 4))

    # raw data
    ax.scatter(xs, ys, color='steelblue', s=60, zorder=3, label='data')
    ax.plot(xs, ys, color='steelblue', alpha=0.25, linewidth=1)

    # skyline
    ax.plot(sky_x, sky_y, color='crimson', linewidth=2,
            marker='o', markersize=7, zorder=4, label='skyline envelope')

    if show_valley:
        val  = envelope.valleyline_envelope(source, tolerance=tolerance)
        val_x, val_y = zip(*val) if val else ([], [])
        ax.plot(val_x, val_y, color='seagreen', linewidth=2,
                marker='o', markersize=7, zorder=4, label='valleyline envelope')

    # get current ticks and if spacing < 1, enforce 1 miniumm
    ticks = ax.get_yticks()
    if len(ticks) >= 2 and (ticks[1] - ticks[0] < 1):
        ymin, ymax = ax.get_ylim()
        int_ticks = np.arange(np.floor(ymin), np.ceil(ymax) + 1, 1)
        ax.set_yticks(int_ticks)

    ax.set_title(title or str(source), fontsize=13)
    ax.set_xlabel('onset')
    ax.set_ylabel('pitch')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.savefig(f'{title}.pdf'). # In case you want this ;)
    plt.show()

    # print summary
    print(f'  raw data ({len(raw)} pts): {[int(y) for _,y in raw]}')
    print(f'  skyline  ({len(sky)} pts): {[int(y) for _,y in sky]}')
    if show_valley:
        val = envelope.valleyline_envelope(source, tolerance=tolerance)
        print(f'  valley   ({len(val)} pts): {[int(y) for _,y in val]}')

Example 1:

`515626738` as a string

In [ ]:
toy_1 = "5156267378"
plot_envelopes(toy_1, title=f'Toy 1: {toy_1}')

Example 2:

`[3, 2, 4, 3, 5, 4, 6, 4, 5, 3, 4, 2, 3]` as a list

In [ ]:
toy_2 = [3, 2, 4, 3, 5, 4, 6, 4, 5, 3, 4, 2, 3]
plot_envelopes(toy_2, title=f'Toy 2: {toy_2}')

Example 3:

Multiple direction changes

In [ ]:
toy_3 = '313131313131424231313131313142536'
plot_envelopes(toy_3, title=f'Toy 3: {toy_3}, multiple direction changes')

---

## <font color='green'> Effect of `tolerance`

`tolerance=0` (default) keeps every point on the exact boundary.
A positive tolerance removes additional points whose dip is shallow,
producing a smoother, coarser envelope.

In [ ]:
noisy = [
    72, 71, 74, 72, 76, 74, 77, 76, 79, 77, 81, 79, 77, 76, 74, 72, 71, 72, 76, 74, 77, 71, 74
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
tolerances = [0, 1, 2]

for ax, tol in zip(axes, tolerances):
    raw = envelope._to_points(noisy)
    xs  = [p[0] for p in raw]
    ys  = [p[1] for p in raw]
    sky = envelope.skyline_envelope(noisy, tolerance=tol)
    sx, sy = zip(*sky)

    ax.plot(xs, ys, 'o-', color='steelblue', alpha=0.3, label='data')
    ax.plot(sx, sy, 'o-', color='crimson', linewidth=2, label=f'skyline env. (tol={tol})')
    ax.set_title(f'{len(sky)} pts retained when tolerance = {tol}.')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Effect of tolerance on skyline envelopes', fontsize=15)
plt.tight_layout()
plt.show()

## <font color='purple'> Exercise: Apply to a Score

Apply this logic to the successive onsets of a score.

- Use your own choice of score, or our suggestion (given by the `url` below)
- Parse the score
    - Hint: `from amads.io.readscore import read_score` works both for a local path and URL.
- Get the pitch and onset pairs.
- Compare your result against the AMADS implementation (which should be identical).

In [ ]:
# Craig Sapp's encodng of Mozart Piano Sonata No.1
url = "https://raw.githubusercontent.com/craigsapp/mozart-piano-sonatas/refs/heads/main/kern/sonata01-1.krn"

## <font color='crimson'> Solution

In [ ]:
from amads.io.readscore import read_score
from amads.core.basics import Note

In [ ]:
parsed_score = read_score(url)

In [ ]:
right_hand_pitch_onset_pairs = [(x.onset, x.key_num) for x in parsed_score.list_all(Note)]
right_hand_pitch_onset_pairs[:69]

In [ ]:
plot_envelopes(
    right_hand_pitch_onset_pairs[3:29],
    show_valley=False,
    title=f'Mozart 279, Movement 1, Right Hand, Extract'
)

Note that using interpolation contour would pick up every change of direction (every note).

In [ ]:
from amads.melody.contour.interpolation_contour import InterpolationContour
o, p = zip(*right_hand_pitch_onset_pairs[:10])
mozart_ic = InterpolationContour(pitches=p, onsets=o)
mozart_ic.plot()

## <font color='purple'> Bonus Exercise: Apply to Score Data, explore tolerance

Below are the first 68 pitch-onset pairs in the Bach BWV1007 Courante example from the book
(up to the first cadence).

Apply the above method and explore the tolerance values.

Which captures the book's reduction best?

In [ ]:
courante_68 = [
    (2.5, 55),
    (3.0, 55),
    (3.5, 50),
    (4.0, 43),
    (4.5, 59),
    (4.75, 60),
    (5.0, 62),
    (5.25, 60),
    (5.5, 59),
    (5.75, 57),
    (6.0, 59),
    (6.5, 50),
    (7.0, 43),
    (7.5, 55),
    (7.75, 57),
    (8.0, 59),
    (8.5, 55),
    (9.0, 52),
    (9.5, 48),
    (10.0, 36),
    (10.5, 57),
    (10.75, 59),
    (11.0, 60),
    (11.25, 59),
    (11.5, 57),
    (11.75, 55),
    (12.0, 54),
    (12.5, 50),
    (13.0, 38),
    (13.5, 50),
    (13.75, 52),
    (14.0, 54),
    (14.25, 55),
    (14.5, 57),
    (14.75, 59),
    (15.0, 60),
    (15.25, 59),
    (15.5, 60),
    (15.75, 57),
    (16.0, 60),
    (16.25, 59),
    (16.5, 60),
    (16.75, 57),
    (17.0, 50),
    (17.25, 60),
    (17.5, 59),
    (17.75, 57),
    (18.0, 59),
    (18.25, 57),
    (18.5, 59),
    (18.75, 55),
    (19.0, 59),
    (19.25, 57),
    (19.5, 59),
    (19.75, 55),
    (20.0, 48),
    (20.25, 59),
    (20.5, 57),
    (20.75, 55),
    (21.0, 54),
    (21.25, 57),
    (21.5, 62),
    (21.75, 50),
    (22.0, 55),
    (22.5, 47),
    (23.0, 38),
    (23.5, 54),
    (24.0, 43),
    (24.0, 55)
]

## <font color='crimson'> Solution

In [ ]:
fig, axes = plt.subplots(3, figsize=(15, 14)) #, sharey=True)
tolerances = [2, 3, 4]

for ax, tol in zip(axes, tolerances):
    raw = envelope._to_points(courante_68)
    xs  = [p[0] for p in raw]
    ys  = [p[1] for p in raw]
    sky = envelope.skyline_envelope(courante_68, tolerance=tol)
    sx, sy = zip(*sky)

    ax.plot(xs, ys, 'o-', color='steelblue', alpha=0.3, label='data')
    ax.plot(sx, sy, 'o-', color='crimson', linewidth=2, label=f'skyline env. (tol={tol})')
    ax.set_title(f'{len(sky)} pts retained when tolerance = {tol}.')
    ax.set_xticks(np.arange(3, max(xs), 3))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(
    "Effect of Tolerance on the Skyline Envelope for Bach's Courante (BWV1007/3, Opening)",
    fontsize=15
)
plt.tight_layout()
# plt.savefig('envelope.pdf') # In case you want this ;)
plt.show()


---
## API summary

| Function            | Returns                | Notes                             |
|---------------------|------------------------|-----------------------------------|
| `skyline_envelope`  | `list[(onset, pitch)]` | upper envelope, exact data points |
| `valleyline`        | `list[(onset, pitch)]` | lower envelope, exact data points |
| `skyline_values`    | `list[float]`          | y-values only                     |
| `valleyline_values` | `list[float]`          | y-values only                     |

Each function takes as arguments:
1. `source` which accepts:
    - `str` of digit characters — e.g. `"313131"`
    - `list` of scalars — onsets synthesised as `1, 2, 3, ...`
    - `list` of `(onset, pitch)` pairs — arbitrary spacing
    - AMADS `Score` object and related (e.g., `Part`) on which the `.find_all(Note)` operation is valid
2. `tolerance` which concerns the 'tolerated' distance from a neighbouring notes (measured in semi-tones):
    - 0 = exact envelope, the default,
    - 1 accepts notes within one ST,
    - likewise for greater values


End

---